# LARNet Training on Google Colab (GPU)
Upload this notebook to Google Colab, then run all cells.
- Runtime → Change runtime type → GPU (T4)
- Estimated training time: ~15 min for 100 epochs

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import time
import os

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## Model Definition (LARNet)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        w = self.squeeze(x).view(b, c)
        w = self.excitation(w).view(b, c, 1, 1)
        return x * w


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=3,
                                   stride=stride, padding=1, groups=in_channels, bias=False)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pointwise(x)
        x = self.bn2(x)
        x = self.relu(x)
        return x


class LARBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv = DepthwiseSeparableConv(in_channels, out_channels, stride=stride)
        self.se = SEBlock(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv(x)
        out = self.se(out)
        out = out + residual
        out = self.relu(out)
        return out


class LARNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.stage1 = nn.Sequential(LARBlock(32, 64, stride=1), LARBlock(64, 64, stride=1))
        self.stage2 = nn.Sequential(LARBlock(64, 128, stride=2), LARBlock(128, 128, stride=1), LARBlock(128, 128, stride=1))
        self.stage3 = nn.Sequential(LARBlock(128, 256, stride=2), LARBlock(256, 256, stride=1), LARBlock(256, 256, stride=1))
        self.stage4 = nn.Sequential(LARBlock(256, 512, stride=2), LARBlock(512, 512, stride=1))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = LARNet(num_classes=10)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,} ({total_params/1e6:.2f}M)')

## Data Loading

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    transforms.RandomErasing(p=0.1),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f'Training samples: {len(trainset)}')
print(f'Test samples: {len(testset)}')

## Training (100 Epochs)

In [ ]:
EPOCHS = 100
LR = 0.05
WEIGHT_DECAY = 5e-4

model = LARNet(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses = []
test_accs = []
best_acc = 0.0
epoch_times = []

print(f'Training LARNet for {EPOCHS} epochs on {device}')
print('='*60)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    t0 = time.time()

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    scheduler.step()
    avg_loss = running_loss / len(trainloader)
    train_acc = 100.0 * correct / total
    train_losses.append(avg_loss)

    # Evaluate
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
    test_acc = 100.0 * test_correct / test_total
    test_accs.append(test_acc)

    elapsed = time.time() - t0
    epoch_times.append(elapsed)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), 'larnet_best_100ep.pth')

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:3d}/{EPOCHS}] loss={avg_loss:.4f} '
              f'train={train_acc:.1f}% test={test_acc:.2f}% '
              f'best={best_acc:.2f}% lr={scheduler.get_last_lr()[0]:.5f} '
              f'time={elapsed:.1f}s')

total_time = sum(epoch_times)
print(f'\n{"="*60}')
print(f'Best test accuracy: {best_acc:.2f}%')
print(f'Total training time: {total_time/60:.1f} min')
print(f'Avg epoch time: {np.mean(epoch_times):.1f}s')

## Training Curves & Download

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, linewidth=1.5)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('LARNet Training Loss (100 epochs, GPU)')
ax1.grid(True, alpha=0.3)

ax2.plot(test_accs, linewidth=1.5, color='green')
ax2.axhline(y=best_acc, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_acc:.2f}%')
ax2.axhline(y=89, color='orange', linestyle=':', alpha=0.5, label='Previous (20ep): 89%')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Test Accuracy (%)')
ax2.set_title('LARNet Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_100ep.png', dpi=150, bbox_inches='tight')
plt.show()

# Save arrays
np.save('test_accs_100ep.npy', np.array(test_accs))
np.save('train_losses_100ep.npy', np.array(train_losses))

print(f'\nFiles saved: larnet_best_100ep.pth, training_curves_100ep.png')

# Download
try:
    from google.colab import files
    files.download('larnet_best_100ep.pth')
    files.download('training_curves_100ep.png')
except:
    print('Not on Colab - files in current directory')